## AutoShop Multi-Agent

Dieses Notebook basiert auf dem [AutoShop MCP-Server](../05-mcp/30-autoshop-mcp-server.ipynb) Notebook welches zuerst ausgeführt werden muss.

Es ergänzt das Original Notebook um Agenten:
1. Er analysiert den Kundenwunsch.
2. Er ruft die bestehenden AutoShop-MCP-Tools auf.
3. Er wählt aus den gefundenen Catalog Items ein passendes Auto.
4. Er ergänzt Demo-Extras und Argumentation mit einer externen KI ohne MCP-Zugriff.
5. Er erstellt eine Markdown-Offerte.


---

### Grundfunktionen aus dem [AutoShop MCP-Server](../05-mcp/30-autoshop-mcp-server.ipynb) Notebook

In [ ]:
%run ~/data/env.py
! cat ~/data/env.py

Verbindung zum OpenAI API 

In [ ]:
%run ~/data/env.py

import subprocess
import json
import traceback
from openai import OpenAI

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

def get_server_url():
    server_ip = subprocess.check_output(
        "cat ~/data/server-ip",
        shell=True,
        text=True
    ).strip()

    node_port = subprocess.check_output(
        "kubectl get service --namespace ms-mcp autoshop-mcp-server "
        "-o=jsonpath='{ .spec.ports[0].nodePort }'",
        shell=True,
        text=True
    ).strip()

    return f"http://{server_ip}:{node_port}/mcp"

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

MCP_URL = get_server_url()

Dann eine kleine Konvertierung von MCP-Tools zu OpenAI-Tools:

In [ ]:
def mcp_tool_to_openai_tool(tool):
    schema = tool.inputSchema or {
        "type": "object",
        "properties": {},
    }

    return {
        "type": "function",
        "name": tool.name,
        "description": getattr(tool, "description", None) or f"MCP Tool {tool.name}",
        "parameters": schema,
    }


def mcp_result_to_text(result):
    parts = []

    for block in result.content:
        if hasattr(block, "text"):
            parts.append(block.text)
        elif hasattr(block, "model_dump"):
            parts.append(json.dumps(block.model_dump(), ensure_ascii=False))
        else:
            parts.append(str(block))

    return "\n".join(parts)


Read-only Toolset. order_delete ist hier bewusst nicht freigegeben.

In [ ]:
READ_ONLY_TOOLS = {
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
}

Der effektive Aufruf

In [ ]:
async def ask_mcp(question: str, allowed_tools=READ_ONLY_TOOLS, max_rounds: int = 8):
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()

            mcp_tools = [
                tool for tool in tools_result.tools
                if tool.name in allowed_tools
            ]

            openai_tools = [
                mcp_tool_to_openai_tool(tool)
                for tool in mcp_tools
            ]

            input_items = [
                {
                    "role": "system",
                    "content": (
                        "Du bist ein Assistent für einen Demo-Shop. "
                        "Nutze die bereitgestellten Tools für Catalog, Customer und Order. "
                        "Erfinde keine Shop-Daten. Frage Daten zuerst über Tools ab. "
                        "Antworte auf Deutsch."
                    ),
                },
                {
                    "role": "user",
                    "content": question,
                },
            ]

            for _ in range(max_rounds):
                response = client.responses.create(
                    model=AI_MODEL,
                    input=input_items,
                    tools=openai_tools,
                )

                function_calls = [
                    item for item in response.output
                    if item.type == "function_call"
                ]

                if not function_calls:
                    return response.output_text

                input_items += response.output

                for call in function_calls:
                    args = json.loads(call.arguments or "{}")

                    tool_result = await session.call_tool(
                        call.name,
                        arguments=args,
                    )

                    input_items.append({
                        "type": "function_call_output",
                        "call_id": call.call_id,
                        "output": mcp_result_to_text(tool_result),
                    })

            return "Abgebrochen: zu viele Tool-Runden."

---

### Agenten 

Externe KI ohne MCP-Zugriff.
- Sie darf keine internen Shop-Daten abrufen.
- Sie darf keine Preise oder Fahrzeuge erfinden.
- Sie ergänzt nur allgemeine Beschreibung, mögliche Extras und Verkaufsargumente.


In [ ]:
def ask_external_ai(question: str):
    response = client.responses.create(
        model=AI_MODEL,
        input=[
            {
                "role": "system",
                "content": (
                    "Du bist eine externe KI ohne Zugriff auf den AutoShop-Catalog. "
                    "Du darfst keine Preise, Verfügbarkeit, IDs oder konkreten Shop-Daten erfinden. "
                    "Du darfst nur allgemeine Zusatzinformationen, mögliche Extras und "
                    "Verkaufsargumente formulieren. Antworte auf Deutsch."
                ),
            },
            {
                "role": "user",
                "content": question,
            },
        ],
    )

    return response.output_text

**Interner Agententeil**

Nutzt ausschliesslich die bestehenden AutoShop-MCP-Tools.
Ziel:
- passende Catalog Items finden
- Preise aus dem Catalog übernehmen
- eine Empfehlung aussprechen
        

In [ ]:
async def internal_car_agent(user_request: str):
    question = f"""
    Du bist ein Auto-Verkaufsagent für den AutoShop.

    Aufgabe:
    1. Analysiere den Kundenwunsch.
    2. Verwende die vorhandenen Catalog-Tools.
    3. Liste passende Fahrzeuge aus dem Catalog auf.
    4. Verwende ausschliesslich Preise aus dem Catalog.
    5. Wähle das am besten passende Fahrzeug aus.
    6. Begründe die Auswahl kurz.
    7. Gib die Antwort strukturiert aus.

    Wichtig:
    - Erfinde keine Fahrzeuge.
    - Erfinde keine Preise.
    - Wenn Eigenschaften fehlen, nutze den Rohinhalt aus dem Catalog.
    - Die Fahrzeugdaten müssen aus dem AutoShop-MCP kommen.

    Kundenwunsch:
    {user_request}
    """

    return await ask_mcp(question)

**Externer Agententeil**

- Ergänzt allgemeine Demo-Informationen.
- Keine Preise.
- Keine internen IDs.
- Keine Verfügbarkeit.

In [ ]:
def external_car_enrichment(user_request: str, internal_result: str):
    question = f"""
    Ein Kunde sucht ein Auto.

    Kundenwunsch:
    {user_request}

    Interne AutoShop-Daten:
    {internal_result}

    Ergänze allgemeine Informationen für ein Verkaufsgespräch:

    - mögliche sinnvolle Extras
    - typische Vorteile dieses Fahrzeugtyps
    - worauf der Kunde achten sollte
    - kurze Verkaufsargumente

    Regeln:
    - Keine Preise erfinden.
    - Keine Shop-Verfügbarkeit erfinden.
    - Keine neuen Fahrzeuge erfinden.
    - Formuliere die Extras als unverbindliche Vorschläge.
    """

    return ask_external_ai(question)

**Erstellt eine Markdown-Offerte aus:**
- internen AutoShop-Daten
- externen Zusatzinformationen

Die Preise und Fahrzeuge müssen aus internal_result stammen.

In [ ]:
def create_markdown_offer(customer_name: str, user_request: str, internal_result: str, external_result: str):
    """

    """
    response = client.responses.create(
        model=AI_MODEL,
        input=[
            {
                "role": "system",
                "content": (
                    "Du erstellst eine Demo-Offerte als Markdown. "
                    "Fahrzeuge, IDs und Preise dürfen ausschliesslich aus den internen AutoShop-Daten stammen. "
                    "Externe Informationen dürfen nur als unverbindliche Zusatzinformationen verwendet werden. "
                    "Die Offerte ist ein Education-Beispiel und nicht rechtsverbindlich. "
                    "Antworte auf Deutsch."
                ),
            },
            {
                "role": "user",
                "content": f"""
                Erstelle eine Markdown-Offerte.

                Kunde:
                {customer_name}

                Kundenwunsch:
                {user_request}

                Interne AutoShop-Daten:
                {internal_result}

                Externe Zusatzinformationen:
                {external_result}

                Struktur der Offerte:

                # Offerte

                ## Kunde

                ## Kundenwunsch

                ## Empfohlenes Fahrzeug

                ## Preis gemäss AutoShop-Catalog

                ## Empfohlene Extras

                ## Begründung

                ## Hinweis

                Der Hinweis muss sagen:
                Diese Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
                """
            },
        ],
    )

    return response.output_text

**Vollständiger Agentenablauf**

Der Wert des Agenten liegt hier darin, dass aus einem offenen Kundenwunsch
ein mehrstufiger Ablauf entsteht:

- interne Daten suchen
- Resultate bewerten
- externe Zusatzinformationen ergänzen
- Markdown-Offerte erzeugen

In [ ]:
async def car_sales_agent(customer_name: str, user_request: str):

    print("1. Interne AutoShop-Daten werden über MCP gesucht...")
    internal_result = await internal_car_agent(user_request)

    print("2. Externe Zusatzinformationen werden ergänzt...")
    external_result = external_car_enrichment(user_request, internal_result)

    print("3. Markdown-Offerte wird erstellt...")
    offer = create_markdown_offer(
        customer_name=customer_name,
        user_request=user_request,
        internal_result=internal_result,
        external_result=external_result,
    )

    return {
        "internal_result": internal_result,
        "external_result": external_result,
        "offer": offer,
    }

---

### Aufruf

In [ ]:
from IPython.display import Markdown, display

result = await car_sales_agent(
    customer_name="Max Muster",
    user_request="Ich suche ein günstiges Auto mit viel Platz für die Familie."
)

display(Markdown(result["offer"]))

---

### Fazit

- Der Agent ist nicht wertvoll, weil er neue Datenquellen hat.
- Der Agent ist wertvoll, weil er mehrere vorhandene Fähigkeiten zu einem zielgerichteten Ablauf kombiniert.
---
- Aus "**Ich suche ein Auto...**" wird durch den Agenten ein vollständiger Mini-Verkaufsprozess:
- Kundenwunsch → AutoShop-Catalog → Fahrzeugauswahl → Zusatzargumente → Markdown-Offerte.